# Section 5 — Miscellaneous (List Comprehensions, Lambdas, Closures, I/O)

**PCAP-31-03 official section** — 9 exam items, 22% of the exam.

Source: [Python Institute PCAP-31-03 syllabus](https://pythoninstitute.org/pcap-exam-syllabus). Explanations below are grounded in the [official Python documentation](https://docs.python.org/3/) (linked per topic) — read the explanation, then run and tweak the code.

---


## 5.1 Build complex lists using list comprehensions

A list comprehension `[expr for item in iterable if condition]` builds a list in one line instead of a `for` loop with `.append()`. The `if` part is optional and filters which items are included. You can nest a comprehension inside another comprehension's expression to build lists of lists (e.g. a small matrix).


In [ ]:
squares = [x**2 for x in range(10)]
print(squares)

evens_squared = [x**2 for x in range(10) if x % 2 == 0]   # filter with `if`
print(evens_squared)

# Equivalent hand-written loop, for comparison:
manual = []
for x in range(10):
    if x % 2 == 0:
        manual.append(x**2)
print(manual == evens_squared)

# Nested comprehension: build a 3x3 grid
grid = [[row * 3 + col for col in range(3)] for row in range(3)]
for row in grid:
    print(row)


## 5.2 Embed lambda functions into code

`lambda arguments: expression` creates a small, anonymous, single-expression function — equivalent to a `def` but limited to exactly one expression (no statements, no multiple lines). Lambdas are most useful as short throwaway functions passed *into* other functions. `map(func, iterable)` applies `func` to every item and returns an iterator of results. `filter(func, iterable)` keeps only the items where `func(item)` is truthy. Both are commonly wrapped in `list(...)` to see their contents.


In [ ]:
square = lambda x: x ** 2          # equivalent to: def square(x): return x ** 2
print(square(5))

add = lambda a, b: a + b
print(add(3, 4))

numbers = [1, 2, 3, 4, 5, 6]
doubled = list(map(lambda x: x * 2, numbers))
print(doubled)

evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)

# A function that ACCEPTS a lambda as an argument (common exam pattern)
def apply_twice(func, value):
    return func(func(value))

print(apply_twice(lambda x: x + 3, 10))   # (10+3)+3 = 16


## 5.3 Define and use closures

A **closure** is a function defined inside another function that "remembers" the variables from its enclosing scope, even after the outer function has finished running. This lets you generate specialized functions on the fly — each call to the outer function creates a fresh, independent closure with its own captured value.


In [ ]:
def make_multiplier(factor):
    def multiplier(x):        # multiplier "closes over" factor from the enclosing scope
        return x * factor
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5), triple(5))    # each remembers its OWN factor, independently

# Proof the outer function has already returned, yet the value is still remembered:
print(double.__closure__[0].cell_contents)   # 2 -- the captured 'factor'


## 5.4 Understand basic input/output terminology

A **stream** is an abstraction for a flow of data in or out of a program; a **file handle** is the object your code uses to interact with that stream (what `open()` returns). Python starts every program with three predefined streams: `sys.stdin` (keyboard input), `sys.stdout` (normal output — what `print()` writes to), and `sys.stderr` (error output, printed separately from stdout so error messages can be seen/redirected independently). **Text mode** (the default) reads/writes `str` and handles encoding/newline translation for you; **binary mode** (`'b'` added to the file mode) reads/writes raw `bytes` with no translation — required for non-text files like images.


In [ ]:
import sys

print('This goes to stdout (normal output).', file=sys.stdout)
print('This goes to stderr (error output).', file=sys.stderr)

# text vs binary mode, side by side
from pathlib import Path
Path('io_demo.bin').write_bytes(b'\x00\x01\x02Hello')

with open('io_demo.bin', 'rb') as f:      # binary mode: returns bytes
    data = f.read()
    print(type(data), data)

Path('io_demo.bin').unlink()


## 5.5 Perform input/output operations

`open(filename, mode)` returns a file object. Modes: `'r'` read (default), `'w'` write (overwrites!), `'a'` append, add `'b'` for binary. If the file doesn't exist, Python raises `FileNotFoundError`, whose `.errno` attribute holds the low-level OS error number (from the standard `errno` module, e.g. `errno.ENOENT`). `.close()` releases the file — but prefer `with open(...) as f:`, which closes automatically even if an exception happens inside the block. `.read()` reads everything (or up to *n* characters/bytes if given a size); `.readline()` reads one line at a time; `.readlines()` reads every line into a list; `.write(data)` writes a string (text mode) or bytes (binary mode). A `bytearray` is a *mutable* sequence of bytes — useful as an in-memory buffer you can read into and modify piece by piece, unlike an immutable `bytes` object.


In [ ]:
import errno
from pathlib import Path

# FileNotFoundError and its .errno
try:
    open('this_file_does_not_exist.txt')
except FileNotFoundError as e:
    print('errno:', e.errno, '== errno.ENOENT?', e.errno == errno.ENOENT)
    print('strerror:', e.strerror)

# Writing then reading, with the safe `with` pattern
with open('pcap_io_demo.txt', 'w') as f:
    f.write('line one\n')
    f.write('line two\n')

with open('pcap_io_demo.txt', 'r') as f:
    print(repr(f.readline()))    # just the first line, newline included
    print(f.readlines())          # the rest, as a list of lines

Path('pcap_io_demo.txt').unlink()

# bytearray as a mutable I/O buffer
buffer = bytearray(5)          # 5 zero bytes: bytearray(b'\x00\x00\x00\x00\x00')
print(buffer)
buffer[0] = 65                  # mutate a single byte in place -- bytes objects can't do this
print(buffer, bytes(buffer))
